# Xử lý ngôn ngữ tự nhiên - CS221.Q21.KHTN

## Demo chương 17 - Sequence Labeling for Parts of Speech and Named Entities

### Nhóm 1:
- Bảo Quý Định Tân - 24520028
- Lê Văn Thức - 24521748
- Lê Phạm Thành Nhân - 24520022

In [1]:
!pip install nltk sklearn-crfsuite scikit-learn

In [2]:
import nltk
import pandas as pd
from nltk.corpus import brown
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Download the brown corpus and universal tagset
nltk.download('brown')
nltk.download('universal_tagset')

# Load tagged sentences from the Brown corpus
print("Loading dataset...")
tagged_sentences = brown.tagged_sents(tagset='universal')

# Significantly increase data size: 40,000 for training, 10,000 for testing
train_size = 40000
test_size = 10000

train_sents = tagged_sentences[:train_size]
test_sents = tagged_sentences[train_size:train_size + test_size]

print(f"Total sentences available: {len(tagged_sentences)}")
print(f"Training sentences: {len(train_sents)}")
print(f"Testing sentences: {len(test_sents)}")

[nltk_data] Downloading package brown to /home/asamai/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package universal_tagset to
[nltk_data]     /home/asamai/nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!


Loading dataset...
Total sentences available: 57340
Training sentences: 40000
Testing sentences: 10000


## Hidden Markov Model (HMM)

In [3]:
from nltk.tag import hmm

print("Training HMM Model (this may take a moment)...")
hmm_model = hmm.HiddenMarkovModelTagger.train(train_sents)

print("Evaluating HMM...")
# Extract true tags
hmm_y_true = [tag for sent in test_sents for _, tag in sent]

# Extract words to feed into the predictor
test_words = [[word for word, tag in sent] for sent in test_sents]

# Predict tags and flatten the list
hmm_y_pred = []
for words in test_words:
    hmm_y_pred.extend([tag for _, tag in hmm_model.tag(words)])

# Calculate metrics
hmm_accuracy = accuracy_score(hmm_y_true, hmm_y_pred)
hmm_macro_f1 = f1_score(hmm_y_true, hmm_y_pred, average='macro')

print(f"HMM Accuracy: {hmm_accuracy:.4f}")
print(f"HMM Macro-F1: {hmm_macro_f1:.4f}")

Training HMM Model (this may take a moment)...
Evaluating HMM...
HMM Accuracy: 0.9332
HMM Macro-F1: 0.8507


## Conditional Random Fields (CRF) 

### Feature Extraction

In [4]:
# Feature extraction for CRF
def word2features(sent, i):
    word = sent[i][0]
    
    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word[-3:]': word[-3:],       # Suffix
        'word[:3]': word[:3],         # Prefix
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
    }
    if i > 0:
        word1 = sent[i-1][0]
        features.update({
            '-1:word.lower()': word1.lower(),
            '-1:word.istitle()': word1.istitle(),
            '-1:word.isupper()': word1.isupper(),
        })
    else:
        features['BOS'] = True 

    if i < len(sent)-1:
        word1 = sent[i+1][0]
        features.update({
            '+1:word.lower()': word1.lower(),
            '+1:word.istitle()': word1.istitle(),
            '+1:word.isupper()': word1.isupper(),
        })
    else:
        features['EOS'] = True

    return features

def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return [label for token, label in sent]

print("Preparing features for CRF...")
X_train = [sent2features(s) for s in train_sents]
y_train = [sent2labels(s) for s in train_sents]

X_test = [sent2features(s) for s in test_sents]
y_test = [sent2labels(s) for s in test_sents]
print("Feature extraction complete!")

Preparing features for CRF...
Feature extraction complete!


### Training and Evaluating the CRF

In [5]:
import sklearn_crfsuite

crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)

print("Training CRF Model...")
crf.fit(X_train, y_train)

print("Evaluating CRF...")
y_pred = crf.predict(X_test)

# Flatten lists for evaluation
flat_y_test = [label for sent in y_test for label in sent]
flat_y_pred = [label for sent in y_pred for label in sent]

crf_accuracy = accuracy_score(flat_y_test, flat_y_pred)
crf_macro_f1 = f1_score(flat_y_test, flat_y_pred, average='macro')

print(f"CRF Accuracy: {crf_accuracy:.4f}")
print(f"CRF Macro-F1: {crf_macro_f1:.4f}")

Training CRF Model...
Evaluating CRF...
CRF Accuracy: 0.9730
CRF Macro-F1: 0.9263


# Results

In [6]:
# Create a summary DataFrame
summary_data = {
    "Model": ["Hidden Markov Model (HMM)", "Conditional Random Field (CRF)"],
    "Accuracy": [hmm_accuracy, crf_accuracy],
    "Macro-F1 Score": [hmm_macro_f1, crf_macro_f1]
}

summary_df = pd.DataFrame(summary_data)
summary_df.set_index("Model", inplace=True)

print("================ MODEL COMPARISON SUMMARY ================")
display(summary_df) # Use print(summary_df) if not using Jupyter

print("\n" + "="*58)
print("Detailed Classification Report for CRF:")
print("="*58)
# Print per-tag metrics for the CRF to see where it excels/struggles
print(classification_report(flat_y_test, flat_y_pred))

================ MODEL COMPARISON SUMMARY ================


,Accuracy,Macro-F1 Score
Model,,
Hidden Markov Model (HMM),0.933179,0.850724
Conditional Random Field (CRF),0.972960,0.926350



Detailed Classification Report for CRF:
              precision    recall  f1-score   support

           .       1.00      1.00      1.00     23492
         ADJ       0.91      0.90      0.90      7644
         ADP       0.96      0.98      0.97     15516
         ADV       0.94      0.92      0.93      8654
        CONJ       0.99      0.99      0.99      4632
         DET       0.99      0.99      0.99     17044
        NOUN       0.96      0.98      0.97     28975
         NUM       0.97      0.98      0.98      1072
        PRON       0.99      0.98      0.99     11216
         PRT       0.95      0.91      0.93      5839
        VERB       0.98      0.98      0.98     27429
           X       0.55      0.45      0.49       116

    accuracy                           0.97    151629
   macro avg       0.93      0.92      0.93    151629
weighted avg       0.97      0.97      0.97    151629



### Nhận xét
Ta thấy rằng CRF có lợi thế hơn khi có thể nhìn thấy được cả câu để trích xuất đặc trưng 